## Konfiguracja i wczytanie danych

In [ ]:
import csv
import pickle

import numpy as np
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence



device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
print(f"Używane urządzenie: {device}")


with open('train.pkl', 'rb') as f:
    train_data_raw = pickle.load(f) # [(sekwencja, klasa), ...]

with open('test_no_target.pkl', 'rb') as f:
    test_data_raw = pickle.load(f)  # [sekwencja1, sekwencja2, ...]

Używane urządzenie: cuda


## vocab - słownik do mapowania akordów na liczby całkowite


In [ ]:
vocab = {}
vocab_idx = 1 # id 0 dla paddingu

for seq, _ in train_data_raw:
    for chord in seq:
        if chord not in vocab:
            vocab[chord] = vocab_idx
            vocab_idx += 1

for seq in test_data_raw:
    for chord in seq:
        if chord not in vocab:
            vocab[chord] = vocab_idx
            vocab_idx += 1

vocab_size = len(vocab) + 1 # +1 dla paddingu


## Przygotowanie Datasetu

In [ ]:
train_data_raw, val_data_raw = train_test_split(train_data_raw, test_size=0.2, shuffle=True, random_state=42)

In [ ]:
class ChordDataset(Dataset):
    def __init__(self, data, is_test=False):
        self.is_test = is_test
        self.data = []

        if not self.is_test:
            for seq, label in data:
                # akord -> id numer
                num_seq = [vocab[chord] for chord in seq]
                self.data.append((torch.tensor(num_seq, dtype=torch.long), label))
        else:
            for seq in data:
                num_seq = [vocab[chord] for chord in seq]
                self.data.append(torch.tensor(num_seq, dtype=torch.long))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = ChordDataset(train_data_raw, is_test=False)
val_dataset = ChordDataset(val_data_raw, is_test=False)
test_dataset = ChordDataset(test_data_raw, is_test=True)


## Obsługa paddingu w batchach

In [ ]:
def pad_collate_train(batch):
    xx, yy = zip(*batch)
    x_lens = [len(x) for x in xx]

    xx_pad = pad_sequence(xx, batch_first=True, padding_value=0) # wypelniamy 0
    return xx_pad, torch.tensor(yy, dtype=torch.long), x_lens

def pad_collate_test(batch):
    x_lens = [len(x) for x in batch]
    xx_pad = pad_sequence(batch, batch_first=True, padding_value=0)
    return xx_pad, x_lens

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=pad_collate_train)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_train)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=pad_collate_test)

dataloaders = {
    "train": train_loader,
    "val": val_loader,
    "test": test_loader
}


## Klasyfikator LSTM

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, num_classes):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # embedding
        # warstwa LSTM
        # warstwa w pelni polaczona

        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, dropout=0.3 if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, x_lens):
        # 1. embedding
        # 2. pakowanie sekwencji
        # 3. przepuszczenie przez LSTM
        # 4. wyciagniecie ostatniego ukrytego stanu z ostatniej warstwy LSTM
        # 5. klasyfikacja

        embedded = self.embedding(x)

        packed_input = pack_padded_sequence(embedded, x_lens, batch_first=True, enforce_sorted=False)
        packed_output, (hn, cn) = self.lstm(packed_input)

        last_hidden = hn[-1, :, :] # wnioski z ost warstwy

        out = self.fc(last_hidden)
        return out

## Inicjalizacja i pętla ucząca

In [ ]:
embedding_dim = 64
hidden_size = 128
num_layers = 2
num_classes = 5 # bach, beethoven, debussy, scarlatti, victoria

model = LSTMClassifier(vocab_size, embedding_dim, hidden_size, num_layers, num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()


epochs = 25

print("Rozpoczęcie treningu...")
model.train()
for epoch in range(epochs):
    total_loss = 0
    correct = 0
    total = 0

    for x, targets, x_lens in train_loader:
        x, targets = x.to(device), targets.to(device)

        optimizer.zero_grad()

        # predykcja
        preds = model(x, x_lens)

        # blad
        loss = loss_fn(preds, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # metryki
        _, predicted = torch.max(preds.data, 1)
        total += targets.size(0)
        correct += (predicted == targets).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch: {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}, Accuracy: {accuracy:.2f}%")


Rozpoczęcie treningu...
Epoch: 1/25, Loss: 1.0936, Accuracy: 59.37%
Epoch: 2/25, Loss: 0.7832, Accuracy: 71.90%
Epoch: 3/25, Loss: 0.6221, Accuracy: 78.36%
Epoch: 4/25, Loss: 0.5520, Accuracy: 80.33%
Epoch: 5/25, Loss: 0.4630, Accuracy: 84.01%
Epoch: 6/25, Loss: 0.4253, Accuracy: 85.13%
Epoch: 7/25, Loss: 0.3818, Accuracy: 86.66%
Epoch: 8/25, Loss: 0.3373, Accuracy: 88.36%
Epoch: 9/25, Loss: 0.2767, Accuracy: 90.30%
Epoch: 10/25, Loss: 0.2334, Accuracy: 92.51%
Epoch: 11/25, Loss: 0.2496, Accuracy: 91.46%
Epoch: 12/25, Loss: 0.2400, Accuracy: 92.00%
Epoch: 13/25, Loss: 0.3060, Accuracy: 89.11%
Epoch: 14/25, Loss: 0.3514, Accuracy: 87.72%
Epoch: 15/25, Loss: 0.2324, Accuracy: 92.07%
Epoch: 16/25, Loss: 0.2898, Accuracy: 90.17%
Epoch: 17/25, Loss: 0.2457, Accuracy: 91.63%
Epoch: 18/25, Loss: 0.1345, Accuracy: 95.61%
Epoch: 19/25, Loss: 0.1113, Accuracy: 96.39%
Epoch: 20/25, Loss: 0.1333, Accuracy: 95.27%
Epoch: 21/25, Loss: 0.0821, Accuracy: 97.58%
Epoch: 22/25, Loss: 0.0615, Accuracy: 98

## Predykcja na zbiorze testowym

In [ ]:
print("Generowanie predykcji na zbiorze testowym...")
model.eval()
all_predictions = []

with torch.no_grad():
    for x, x_lens in test_loader:
        x = x.to(device)
        preds = model(x, x_lens)
        _, predicted_classes = torch.max(preds.data, 1)
        all_predictions.extend(predicted_classes.cpu().tolist())

# zapis do pliku
with open('pred.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for pred in all_predictions:
        writer.writerow([pred])

print("Zapisano wyniki w 'pred.csv'.")

Generowanie predykcji na zbiorze testowym...
Zapisano wyniki w 'pred.csv'.
